In [9]:
import torch
import random
import math
from torch import nn

In [73]:
def train_model(model, train_input, train_target, mini_batch_size, epochs):
    
    criterion = nn.MSELoss()
    eta = 1e-1 / mini_batch_size

    for e in range(epochs):
        sum_loss = 0
        for b in range(0, train_input.size(0), mini_batch_size):
            output = model(train_input.narrow(0, b, mini_batch_size))
            loss = criterion(output, train_target.narrow(0, b, mini_batch_size))
            model.zero_grad()
            loss.backward()
            sum_loss = sum_loss + loss.item()
            with torch.no_grad():
                for p in model.parameters():
                    p -= eta * p.grad
                    
        print(e, sum_loss)

In [74]:
# generate 2d points in [0,1] squared, targets are 0 if point inside the circle of squared radius 1/2pi and 1 outside.
# return coordinates and target tensors, both of size Nx2, plus classes tensor of size Nx1
def generate_points(nb):
    inputs = torch.empty((nb, 2))
    targets = torch.empty((nb, 2))
    classes = torch.empty((nb, 1))
    for i in range(nb):
        x = random.uniform(0,1) - 0.5
        y = random.uniform(0,1) - 0.5
        t = int(2 * math.pi * (pow(x, 2) + pow(y, 2)) < 1)
        inputs[i] = torch.tensor([x + 0.5, y + 0.5])
        targets[i,t] = 1
        targets[i,1-t] = 0
        classes[i] = t
    return inputs, targets, classes


In [81]:
# accuracy for classes prediction
def accuracy(model, inputs, classes):
    
    nb_samples = inputs.shape[0]
    pred = model(inputs)
    _, pred_classes = pred.max(1)
    
    nb_errors = (pred_classes - classes[:,0]).type(torch.BoolTensor).sum().item()
    return (nb_samples - nb_errors) / nb_samples

In [77]:
model = nn.Sequential(
    nn.Linear(2, 25), 
    nn.ReLU(), 
    nn.Linear(25, 25),
    nn.ReLU(),
    nn.Linear(25, 25),
    nn.ReLU(), 
    nn.Linear(25, 2))

In [79]:
train_model(model, input, target, 2, 25)

0 126.13024330884218
1 122.49055521190166
2 107.11639623530209
3 66.31507313763723
4 51.70351446994391
5 46.52351683746383
6 42.280137307883706
7 38.64792177657364
8 36.96896120257588
9 35.12068442375676
10 33.47406784705163
11 32.5975231682969
12 32.25277213957088
13 30.13909637279903
14 29.619116340190885
15 28.539376173148412
16 28.100793222874927
17 26.86246534501697
18 27.31573754492092
19 26.431027301490758
20 26.483320956427633
21 25.12204932862278
22 24.3897690854119
23 24.536916257756275
24 24.266884170942376


In [82]:
accuracy(model, input, classes)

0.893